In [1]:
import pickle
import pandas as pd 
import numpy as np
import torch
import faiss
import os
import re
import requests
import pyreadr
import tempfile
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from transformers import pipeline
import warnings
warnings.filterwarnings("ignore")

# Load the data

* ### QOF and Snomed datasets

In [ ]:
# Qof 
file_id = ''
url = f''
# Snomed
file_id2 = ''
url2 = f''

In [3]:
# Load Qof dataset
df_qof = pd.read_excel(
    url,
    sheet_name="Expanded Cluster List",
    engine="openpyxl",
    skiprows=14,
    dtype={'SNOMED concept ID': 'object'}
)
# Load Snomed dataset
df_snomed = pd.read_excel(url2)

* ### icd10_usage and opcs4_usage datasets

In [ ]:
# Load dataset from GitHub
datasets = {}

for file in ["icd10_usage.rda", "opcs4_usage.rda"]:
    print(f"Downloading {file}...")
    
    url = f""
    r = requests.get(url)
    
    tmp = tempfile.NamedTemporaryFile(suffix=".rda", delete=False)
    tmp.write(r.content)
    tmp.close()

    data = pyreadr.read_r(tmp.name)
    df = list(data.values())[0]
    df.columns = df.columns.str.strip()

    datasets[file] = df
    os.remove(tmp.name)

In [5]:
# Load icd10 and opcs4 datasets
icd10_usage = datasets["icd10_usage.rda"]
opcs4_usage = datasets["opcs4_usage.rda"]

In [6]:
# Visualisation icd10 dataset
icd10_usage.head(2)

,start_date,end_date,icd10_code,description,usage
0,2024-04-01,2025-03-31,A000,"Cholera due to Vibrio cholerae 01, biovar chol...",9
1,2024-04-01,2025-03-31,A009,"Cholera, unspecified",40


In [4]:
# Create source column 
icd10_usage["data_source"] = "OpenCodeCounts_Oxford"

In [5]:
# Visualisation icd10 dataset
icd10_usage.head()

,start_date,end_date,icd10_code,description,usage,data_source
0,2024-04-01,2025-03-31,A000,"Cholera due to Vibrio cholerae 01, biovar chol...",9,OpenCodeCounts_Oxford
1,2024-04-01,2025-03-31,A009,"Cholera, unspecified",40,OpenCodeCounts_Oxford
2,2024-04-01,2025-03-31,A010,Typhoid fever,896,OpenCodeCounts_Oxford
3,2024-04-01,2025-03-31,A011,Paratyphoid fever A,101,OpenCodeCounts_Oxford
4,2024-04-01,2025-03-31,A012,Paratyphoid fever B,14,OpenCodeCounts_Oxford


In [6]:
icd10_usage.isna().sum()

start_date     0
end_date       0
icd10_code     1
description    3
usage          0
data_source    0
dtype: int64

In [7]:
# Visualisation opcs4 dataset
opcs4_usage.head()

,start_date,end_date,opcs4_code,description,usage
0,2024-04-01,2025-03-31,A011,Hemispherectomy,17
1,2024-04-01,2025-03-31,A012,Total lobectomy of brain,29
2,2024-04-01,2025-03-31,A013,Partial lobectomy of brain,126
3,2024-04-01,2025-03-31,A018,Other specified major excision of tissue of brain,25
4,2024-04-01,2025-03-31,A019,Unspecified major excision of tissue of brain,6


In [8]:
# Create source column 
opcs4_usage["data_source"] = "OpenCodeCounts_Oxford"

In [9]:
# Visualisation opcs4 dataset
opcs4_usage.head(3)

,start_date,end_date,opcs4_code,description,usage,data_source
0,2024-04-01,2025-03-31,A011,Hemispherectomy,17,OpenCodeCounts_Oxford
1,2024-04-01,2025-03-31,A012,Total lobectomy of brain,29,OpenCodeCounts_Oxford
2,2024-04-01,2025-03-31,A013,Partial lobectomy of brain,126,OpenCodeCounts_Oxford


In [10]:
opcs4_usage.isna().sum()

start_date     0
end_date       0
opcs4_code     0
description    0
usage          0
data_source    0
dtype: int64

In [11]:
# Visualise Qod data 
df_qof.head(3)

,Coding ID,Cluster ID,Cluster description,SNOMED concept ID,Code description,Type of inclusion (in code string),Included under,Active status
0,SNOMED CT,4IN1VACDRUG_COD,"4-in-1 Diphtheria, tetanus, pertussis and poli...",8.153411e+15,"Diphtheria / Tetanus / Pertussis (acellular, c...",Drug Refset,^72391000001101,1.0
1,SNOMED CT,4IN1VACDRUG_COD,"4-in-1 Diphtheria, tetanus, pertussis and poli...",8.882411e+15,Repevax vaccine suspension for injection 0.5ml...,Drug Refset,^72391000001101,1.0
2,SNOMED CT,4IN1VACDRUG_COD,"4-in-1 Diphtheria, tetanus, pertussis and poli...",8.888711e+15,Generic Repevax vaccine suspension for injecti...,Drug Refset,^72391000001101,1.0


In [12]:
# Create a Type Field
df_qof["Type"] = df_qof["Code description"].str.extract(r"\(([^()]*)\)\s*$")[0].fillna('Unknown').str.strip()

In [13]:
# Remove spaces in column names
df_qof.columns = df_qof.columns.str.replace(' ', '_')
# Create source column 
df_qof["data_source"] = "QoF_business_rules"

In [14]:
# Visualise Qod data 
df_qof.head(3)

,Coding_ID,Cluster_ID,Cluster_description,SNOMED_concept_ID,Code_description,Type_of_inclusion_(in_code_string),Included_under,Active_status,Type,data_source
0,SNOMED CT,4IN1VACDRUG_COD,"4-in-1 Diphtheria, tetanus, pertussis and poli...",8.153411e+15,"Diphtheria / Tetanus / Pertussis (acellular, c...",Drug Refset,^72391000001101,1.0,product,QoF_business_rules
1,SNOMED CT,4IN1VACDRUG_COD,"4-in-1 Diphtheria, tetanus, pertussis and poli...",8.882411e+15,Repevax vaccine suspension for injection 0.5ml...,Drug Refset,^72391000001101,1.0,product,QoF_business_rules
2,SNOMED CT,4IN1VACDRUG_COD,"4-in-1 Diphtheria, tetanus, pertussis and poli...",8.888711e+15,Generic Repevax vaccine suspension for injecti...,Drug Refset,^72391000001101,1.0,product,QoF_business_rules


In [15]:
df_qof.isna().sum()

Coding_ID                             0
Cluster_ID                            0
Cluster_description                   0
SNOMED_concept_ID                     1
Code_description                      1
Type_of_inclusion_(in_code_string)    0
Included_under                        0
Active_status                         1
Type                                  0
data_source                           0
dtype: int64

In [16]:
# Visualise snomed data
df_snomed.head(2)

,SNOMED_Concept_ID,Description,Usage,Active_at_Start,Active_at_End
0,279991000000102,Short message service text message sent to pat...,427336750,1,1
1,184103008,Patient telephone number (observable entity),277158090,1,1


In [17]:
# Create a Type Field
df_snomed["Type"] = df_snomed["Description"].str.extract(r"\(([^()]*)\)\s*$")[0].fillna('Unknown').str.strip()

In [18]:
# Create source column 
df_snomed["data_source"] = "NHS_England_SNOMED_Code_Usage_2024-25"

In [19]:
# visualise the data
df_snomed.head(2)

,SNOMED_Concept_ID,Description,Usage,Active_at_Start,Active_at_End,Type,data_source
0,279991000000102,Short message service text message sent to pat...,427336750,1,1,procedure,NHS_England_SNOMED_Code_Usage_2024-25
1,184103008,Patient telephone number (observable entity),277158090,1,1,observable entity,NHS_England_SNOMED_Code_Usage_2024-25


In [20]:
# find null value
df_snomed.isna().sum()

SNOMED_Concept_ID    0
Description          0
Usage                0
Active_at_Start      0
Active_at_End        0
Type                 0
data_source          0
dtype: int64

In [21]:
# Visualise the data 
print(icd10_usage.shape)
print(opcs4_usage.shape)
print(df_qof.shape)
print(df_snomed.shape)

(147483, 6)
(116680, 6)
(24980, 10)
(158567, 7)


### Create Text Field for NLP / RAG

- icd10_usage 

In [22]:
# Prepare data for embedding
icd10_usage["retrieval_text"] = (
    "code: " + icd10_usage["icd10_code"].fillna("").astype(str) + " | " +
    "term: " + icd10_usage["description"].fillna("").astype(str) + " | " +
    "usage: " + icd10_usage["usage"].fillna("").astype(str) + " | " +
    "start_date: " + icd10_usage["start_date"].fillna("").astype(str) + " | " +
    "end_date: " + icd10_usage["end_date"].fillna("").astype(str) + " | " +
    "source: " + icd10_usage["data_source"].fillna("").astype(str) 
)

In [23]:
# Visualise the data
icd10_usage.head()

,start_date,end_date,icd10_code,description,usage,data_source,retrieval_text
0,2024-04-01,2025-03-31,A000,"Cholera due to Vibrio cholerae 01, biovar chol...",9,OpenCodeCounts_Oxford,code: A000 | term: Cholera due to Vibrio chole...
1,2024-04-01,2025-03-31,A009,"Cholera, unspecified",40,OpenCodeCounts_Oxford,"code: A009 | term: Cholera, unspecified | usag..."
2,2024-04-01,2025-03-31,A010,Typhoid fever,896,OpenCodeCounts_Oxford,code: A010 | term: Typhoid fever | usage: 896 ...
3,2024-04-01,2025-03-31,A011,Paratyphoid fever A,101,OpenCodeCounts_Oxford,code: A011 | term: Paratyphoid fever A | usage...
4,2024-04-01,2025-03-31,A012,Paratyphoid fever B,14,OpenCodeCounts_Oxford,code: A012 | term: Paratyphoid fever B | usage...


In [24]:
# create a new pd
retrieval_icd10_usage = pd.DataFrame(icd10_usage["retrieval_text"]) 

In [25]:
# Visualise the data
retrieval_icd10_usage.head()

,retrieval_text
0,code: A000 | term: Cholera due to Vibrio chole...
1,"code: A009 | term: Cholera, unspecified | usag..."
2,code: A010 | term: Typhoid fever | usage: 896 ...
3,code: A011 | term: Paratyphoid fever A | usage...
4,code: A012 | term: Paratyphoid fever B | usage...


- opcs4_usage

In [26]:
# Prepare data for embedding 
opcs4_usage["retrieval_text"] = (
    "code: " + opcs4_usage["opcs4_code"].fillna("").astype(str) + " | " +
    "term: " + opcs4_usage["description"].fillna("").astype(str) + " | " +
    "usage: " + opcs4_usage["usage"].fillna("").astype(str) + " | " +
    "start_date: " + opcs4_usage["start_date"].fillna("").astype(str) + " | " +
    "end_date: " + opcs4_usage["end_date"].fillna("").astype(str) + " | " +
    "source: " + opcs4_usage["data_source"].fillna("").astype(str) 
)

In [27]:
# Visualise the data
opcs4_usage.head()

,start_date,end_date,opcs4_code,description,usage,data_source,retrieval_text
0,2024-04-01,2025-03-31,A011,Hemispherectomy,17,OpenCodeCounts_Oxford,code: A011 | term: Hemispherectomy | usage: 17...
1,2024-04-01,2025-03-31,A012,Total lobectomy of brain,29,OpenCodeCounts_Oxford,code: A012 | term: Total lobectomy of brain | ...
2,2024-04-01,2025-03-31,A013,Partial lobectomy of brain,126,OpenCodeCounts_Oxford,code: A013 | term: Partial lobectomy of brain ...
3,2024-04-01,2025-03-31,A018,Other specified major excision of tissue of brain,25,OpenCodeCounts_Oxford,code: A018 | term: Other specified major excis...
4,2024-04-01,2025-03-31,A019,Unspecified major excision of tissue of brain,6,OpenCodeCounts_Oxford,code: A019 | term: Unspecified major excision ...


In [28]:
# create a new pd
retrieval_opcs4_usage = pd.DataFrame(opcs4_usage["retrieval_text"]) 

In [29]:
# Visualise the data
retrieval_opcs4_usage.head()

,retrieval_text
0,code: A011 | term: Hemispherectomy | usage: 17...
1,code: A012 | term: Total lobectomy of brain | ...
2,code: A013 | term: Partial lobectomy of brain ...
3,code: A018 | term: Other specified major excis...
4,code: A019 | term: Unspecified major excision ...


- df_qof

In [30]:
# Prepare data for embedding 
df_qof["retrieval_text"] = (
    "code: " + df_qof["SNOMED_concept_ID"].fillna("").astype(str) + " | " +
    "term: " + df_qof["Code_description"].fillna("").astype(str) + " | " +
    "Coding_ID: " + df_qof["Coding_ID"].fillna("").astype(str) + " | " +
    "Cluster_ID: " + df_qof["Cluster_ID"].fillna("").astype(str) + " | " +
    "Cluster_description: " + df_qof["Cluster_description"].fillna("").astype(str) + " | " +
    "Type_of_inclusion_(in_code_string): " + df_qof["Type_of_inclusion_(in_code_string)"].fillna("").astype(str) + " | " +
    "Included_under: " + df_qof["Included_under"].fillna("").astype(str) + " | " +
    "Active_status: " + df_qof["Active_status"].fillna("").astype(str) + " | " +
    "Type: " + df_qof["Type"].fillna("").astype(str) + " | " +
    "source: " + df_qof["data_source"].fillna("").astype(str) 
)

In [31]:
# Visualise the data
df_qof.head()

,Coding_ID,Cluster_ID,Cluster_description,SNOMED_concept_ID,Code_description,Type_of_inclusion_(in_code_string),Included_under,Active_status,Type,data_source,retrieval_text
0,SNOMED CT,4IN1VACDRUG_COD,"4-in-1 Diphtheria, tetanus, pertussis and poli...",8.153411e+15,"Diphtheria / Tetanus / Pertussis (acellular, c...",Drug Refset,^72391000001101,1.0,product,QoF_business_rules,code: 8153411000001106.0 | term: Diphtheria / ...
1,SNOMED CT,4IN1VACDRUG_COD,"4-in-1 Diphtheria, tetanus, pertussis and poli...",8.882411e+15,Repevax vaccine suspension for injection 0.5ml...,Drug Refset,^72391000001101,1.0,product,QoF_business_rules,code: 8882411000001103.0 | term: Repevax vacci...
2,SNOMED CT,4IN1VACDRUG_COD,"4-in-1 Diphtheria, tetanus, pertussis and poli...",8.888711e+15,Generic Repevax vaccine suspension for injecti...,Drug Refset,^72391000001101,1.0,product,QoF_business_rules,code: 8888711000001107.0 | term: Generic Repev...
3,SNOMED CT,4IN1VACDRUG_COD,"4-in-1 Diphtheria, tetanus, pertussis and poli...",2.626721e+16,Boostrix-IPV suspension for injection 0.5ml pr...,Drug Refset,^72391000001101,1.0,product,QoF_business_rules,code: 2.62672110000011e+16 | term: Boostrix-IP...
4,SNOMED CT,4IN1VACDRUG_COD,"4-in-1 Diphtheria, tetanus, pertussis and poli...",8.149311e+15,Repevax vaccine suspension for injection 0.5ml...,Drug Refset,^72391000001101,0.0,product,QoF_business_rules,code: 8149311000001104.0 | term: Repevax vacci...


In [32]:
# create a new pd
retrieval_df_qof = pd.DataFrame(df_qof["retrieval_text"]) 

In [33]:
# Visualise the data
retrieval_df_qof.head()

,retrieval_text
0,code: 8153411000001106.0 | term: Diphtheria / ...
1,code: 8882411000001103.0 | term: Repevax vacci...
2,code: 8888711000001107.0 | term: Generic Repev...
3,code: 2.62672110000011e+16 | term: Boostrix-IP...
4,code: 8149311000001104.0 | term: Repevax vacci...


- df_snomed

In [34]:
# Prepare data for embedding 
df_snomed["retrieval_text"] = (
    "code: " + df_snomed["SNOMED_Concept_ID"].fillna("").astype(str) + " | " +
    "term: " + df_snomed["Description"].fillna("").astype(str) + " | " +
    "Usage: " + df_snomed["Usage"].fillna("").astype(str) + " | " +
    "Active_at_Start: " + df_snomed["Active_at_Start"].fillna("").astype(str) + " | " +
    "Active_at_End: " + df_snomed["Active_at_End"].fillna("").astype(str) + " | " +
    "Type: " + df_snomed["Type"].fillna("").astype(str) + " | " +
    "source: " + df_snomed["data_source"].fillna("").astype(str) 
)

In [35]:
# Visualise the data
df_snomed.head()

,SNOMED_Concept_ID,Description,Usage,Active_at_Start,Active_at_End,Type,data_source,retrieval_text
0,279991000000102,Short message service text message sent to pat...,427336750,1,1,procedure,NHS_England_SNOMED_Code_Usage_2024-25,code: 279991000000102 | term: Short message se...
1,184103008,Patient telephone number (observable entity),277158090,1,1,observable entity,NHS_England_SNOMED_Code_Usage_2024-25,code: 184103008 | term: Patient telephone numb...
2,423876004,Clinical document (record artifact),88284520,1,1,record artifact,NHS_England_SNOMED_Code_Usage_2024-25,code: 423876004 | term: Clinical document (rec...
3,72313002,Systolic arterial pressure (observable entity),70592440,1,1,observable entity,NHS_England_SNOMED_Code_Usage_2024-25,code: 72313002 | term: Systolic arterial press...
4,1091811000000102,Diastolic arterial pressure (observable entity),70557150,1,1,observable entity,NHS_England_SNOMED_Code_Usage_2024-25,code: 1091811000000102 | term: Diastolic arter...


In [36]:
# create a new pandas dataframe
retrieval_df_snomed = pd.DataFrame(df_snomed["retrieval_text"]) 

In [37]:
# Visualise the data
retrieval_df_snomed.head()

,retrieval_text
0,code: 279991000000102 | term: Short message se...
1,code: 184103008 | term: Patient telephone numb...
2,code: 423876004 | term: Clinical document (rec...
3,code: 72313002 | term: Systolic arterial press...
4,code: 1091811000000102 | term: Diastolic arter...


### Combine the dataset

In [38]:
# combine the datasets
clinical_data = pd.concat([
    retrieval_icd10_usage,
    retrieval_opcs4_usage,
    retrieval_df_qof,
    retrieval_df_snomed
], ignore_index=True)

In [39]:
# visualise the first row
clinical_data['retrieval_text'][0]

'code: A000 | term: Cholera due to Vibrio cholerae 01, biovar cholerae | usage: 9 | start_date: 2024-04-01 | end_date: 2025-03-31 | source: OpenCodeCounts_Oxford'

In [40]:
# Visualise the shape 
clinical_data.shape

(447710, 1)

In [41]:
# Convert data into a list 
retrieval_text = clinical_data["retrieval_text"].tolist()

### load the model 

In [ ]:
# load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

### Generate embeddings

In [ ]:
# generate embeddings
embeddings = model.encode(
    retrieval_text,
    show_progress_bar=True
)

In [ ]:
# Print eembeddings shape
print(embeddings.shape)

#### Save Embeddings 

In [ ]:
# save the embedding 
with open("clinical_embeddings.pkl", "wb") as f:
    pickle.dump(embeddings, f)

#### Load embeddings

In [45]:
# Load the embedding
with open("codelist_4data_embeddings.pkl", "rb") as f:
    embeddings = pickle.load(f)

In [46]:
# Embeddig vector dimension
dimension = embeddings.shape[1]
dimension

384

In [47]:
# Print eembeddings shape
print(embeddings.shape)

(447710, 384)


### Connect to DB and Load the embedding to DB

In [ ]:
# connect to the database
conn = psycopg2.connect(
    dbname="clinical_db",
    user="postgres",
    password="",
    host="localhost",
    port=""
)
# A cursor to interact with the database.
cur = conn.cursor()

In [50]:
# Insert the embedding into the database
for text, emb in zip(retrieval_text, embeddings):
    cur.execute(
        """
        INSERT INTO codelist_4data_vectors (retrieval_text, embedding)
        VALUES (%s,%s)
        """,
        (text, emb.tolist())
    )
conn.commit()

# Search from the Database without running the entire process

In [45]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from langchain_community.llms import Ollama
from rank_bm25 import BM25Okapi
import numpy as np
import pandas as pd 
import psycopg2

In [ ]:
# Connect to the database
conn = psycopg2.connect(
    dbname="clinical_db",
    user="postgres",
    password="",
    host="localhost",
    port=""
)

cur = conn.cursor()

In [47]:
# load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### LLM 

In [48]:
llm = Ollama(
    model="gemma3:4b",
    temperature=0.3
)

### Query for LLM

In [180]:
# query
query = "I need to investigate pathologies related to high cholesterol"

### Expand the query using your LLM

In [181]:
# Expansion of the query using the LLM  to help improve retrieval accuracy.

expansion_prompt = f"""
You are a clinical terminology expert.

Generate alternative medical search queries with different characteristics:
1. Exact synonyms
2. Clinical abbreviations
3. Related broader concepts
4. Specific subtypes
5. Common misspellings or variations
    
Original query: {query}
Return as numbered list:
"""

expanded = llm.invoke(expansion_prompt)

### Convert expanded queries to a list

In [183]:
# Convert expanded queries to a list
raw_queries = [query] + expanded.split("\n")
expanded_queries = []
seen = set()

for q in raw_queries:
    q = q.strip()
    q = re.sub(r"^\d+[\).\-\s]*", "", q)
    if q and q.lower() not in seen:
        expanded_queries.append(q)
        seen.add(q.lower())


### Run pgvector search for each query

In [184]:
# Run pgvector search for each query
all_candidates = []   # initialize list first
for q in expanded_queries:

    q_embedding = model.encode([q], convert_to_numpy=True)

    # convert to pgvector format
    vector_str = "[" + ",".join(map(str, q_embedding[0])) + "]"

    cur.execute("""
        SELECT retrieval_text,
        embedding <=> %s::vector AS distance
        FROM codelist_4data_vectors
        ORDER BY embedding <=> %s::vector
        LIMIT 20
    """, (vector_str, vector_str))

    rows = cur.fetchall()
    all_candidates.extend(rows)

### Remove duplicates

In [185]:
best_candidates = {}

for text, dist in all_candidates:
    if text not in best_candidates or dist < best_candidates[text]:
        best_candidates[text] = dist

unique_texts = list(best_candidates.keys())

### Apply hybrid ranking

In [186]:
# run TF-IDF + BM25 + semantic scoring on unique_texts

# Apply your TF-IDF ranking
# create vectorizer
tfidf = TfidfVectorizer()
# build TF-IDF matrix
tfidf_matrix = tfidf.fit_transform(unique_texts)
# convert query to vector
query_vec = tfidf.transform([query])

# compute similarity
scores_tfidf = cosine_similarity(query_vec, tfidf_matrix)[0]

# BM25:

tokenized_corpus = [text.split() for text in unique_texts]
bm25 = BM25Okapi(tokenized_corpus)
bm25_scores = bm25.get_scores(query.split())

# Normalize bm25_scores:
def normalize(x):
    x = np.array(x)
    min_val = np.min(x)
    max_val = np.max(x)

    if max_val == min_val:
        return np.ones_like(x)

    return (x - min_val) / (max_val - min_val)
    
tfidf_norm = normalize(scores_tfidf)
bm25_norm = normalize(bm25_scores)

### Combine scores (hybrid ranking)

In [187]:
semantic_scores = np.array([1 - best_candidates[text] for text in unique_texts])
semantic_norm = normalize(semantic_scores)

results = []

for i, text in enumerate(unique_texts):

    final_score = (
        0.5 * semantic_norm[i] +
        0.25 * tfidf_norm[i] +
        0.25 * bm25_norm[i]
    )

    results.append({
        "retrieval_text": text,
        "semantic_score": f"{semantic_norm[i]:.3f}",
        "scores_tfidf": f"{tfidf_norm[i]:.3f}",
        "bm25_norm": f"{bm25_norm[i]:.3f}",
        "final_score": f"{final_score:.3f}"
    })

results = sorted(results, key=lambda x: float(x["final_score"]), reverse=True)

In [188]:
# put the result into pandas dataframe
codelists = pd.DataFrame(results)

In [189]:
# Visualise
codelists.head()

,retrieval_text,semantic_score,scores_tfidf,bm25_norm,final_score
0,code: 397608007 | term: Finding related to car...,0.496,1.000,1.000,0.748
1,code: 403830007 | term: Familial hypercholeste...,0.938,0.284,0.350,0.628
2,code: 403829002 | term: Familial hypercholeste...,0.926,0.281,0.350,0.621
3,code: 767133009.0 | term: Familial hypercholes...,1.000,0.153,0.250,0.601
4,code: 767133009 | term: Familial hypercholeste...,0.961,0.181,0.298,0.601


In [190]:
codelists.shape

(404, 5)

In [191]:
# Save the list found 
codelists.to_csv("codelists.csv", index=False )

### Send top results to the LLM for explanation

In [192]:
# Send top results to the LLM for explanation
all_rows = []

for r in results[:50]:

    prompt = f"""
    You are assisting NICE analysts.
    
    Retrieval text: {r['retrieval_text']}

    Research question:
    {query}

    Return ONLY JSON.

    {{
      "code": "",
      "term": "",
      "source": "",
      "type": "",
      "Explain why this code is relevant and summarise": ""
    }}

    """

    response = llm.invoke(prompt)
    text = response if isinstance(response, str) else response.content

    import re, json
    blocks = re.findall(r"```json\s*(.*?)\s*```", text, re.S)

    for block in blocks:
        try:
            data = json.loads(block)

            if isinstance(data, list):
                all_rows.extend(data)
            elif isinstance(data, dict):
                all_rows.append(data)

        except Exception as e:
            print("Parse error:", e)

# convert into a pandas dataframe
df = pd.DataFrame(all_rows)

In [193]:
# Visualise the data 
df.head(10)

,code,term,source,type,Explain why this code is relevant and summarise
0,397608007,Finding related to care planning (finding),NHS_England_SNOMED_Code_Usage_2024-25,finding,This code represents a finding related to care...
1,403830007,Familial hypercholesterolemia due to homozygou...,NHS_England_SNOMED_Code_Usage_2024-25,disorder,This code represents Familial Hypercholesterol...
2,403829002,Familial hypercholesterolemia due to heterozyg...,NHS_England_SNOMED_Code_Usage_2024-25,disorder,This code represents Familial Hypercholesterol...
3,767133009.0,Familial hypercholesterolemia co-occurrent and...,QoF_business_rules,disorder,This code specifically identifies Familial Hyp...
4,767133009,Familial hypercholesterolemia co-occurrent and...,NHS_England_SNOMED_Code_Usage_2024-25,disorder,This code specifically describes Familial Hype...
5,166831007,Serum cholesterol very high (finding),NHS_England_SNOMED_Code_Usage_2024-25,finding,This code represents the finding of 'Serum cho...
6,442234001,Serum cholesterol borderline high (finding),NHS_England_SNOMED_Code_Usage_2024-25,finding,This code represents 'Serum cholesterol border...
7,403831006,Familial hypercholesterolemia due to genetic d...,NHS_England_SNOMED_Code_Usage_2024-25,disorder,This code represents Familial Hypercholesterol...
8,403829002.0,Familial hypercholesterolemia due to heterozyg...,QoF_business_rules,disorder,This code directly relates to high cholesterol...
9,403830007.0,Familial hypercholesterolemia due to homozygou...,QoF_business_rules,disorder,This code directly relates to high cholesterol...


In [194]:
# Save the list found 
df.to_csv("ListResult.csv", index=False )